# Local Benchmark Report: AlexNet Performance Analysis (Kaggle T4 Environment)

This notebook documents the implementation, inference validation, and training performance benchmarking of the classic **AlexNet** architecture on a Kaggle **NVIDIA Tesla T4** instance. The evaluation bridges empirical metrics with theoretical limits and historical baselines from the original 2012 paper.

---

## 1. Notebook Execution Pipeline

The notebook transitions through three distinct phases:
1. **Inference Verification:** Loads a pre-trained PyTorch model, passes a test image, and validates the output classification distribution to ensure structural fidelity.
2. **Raw Throughput Inference Benchmarking:** Isolates the forward pass math using standard float32 precision under a tight loop (`100` iterations, batch size `32`) using `torch.cuda.synchronize()` to ensure precise timing hardware profiling.
3. **Optimized Training Stress-Test:** Measures actual backward propagation overhead using a 5-epoch training loop on the CIFAR-10 dataset upsampled to $224 \times 224$ to simulate the math complexity of the full ImageNet workload while remaining completely within the disk/RAM constraints of the environment.

---

## 2. Experimental Output Analysis

### Phase 1: Image Classification Validation

In [1]:
import torch
import torchvision.models as models
import torchvision.transforms as transforms
from PIL import Image
import urllib.request
import os

# 1. Load the pre-trained AlexNet model
# Weights choice 'DEFAULT' gives the best available pre-trained ImageNet weights
print("Loading AlexNet model...")
model = models.alexnet(weights=models.AlexNet_Weights.DEFAULT)
model.eval()  # Set to evaluation mode (turns off dropout, etc.)

# 2. Download ImageNet labels (if not already local)
labels_url = "https://raw.githubusercontent.com/pytorch/hub/master/imagenet_classes.txt"
if not os.path.exists("imagenet_classes.txt"):
    urllib.request.urlretrieve(labels_url, "imagenet_classes.txt")

with open("imagenet_classes.txt", "r") as f:
    categories = [s.strip() for s in f.readlines()]

# 3. Download a sample image (or swap this with your local image path)
img_url = "https://github.com/pytorch/hub/raw/master/images/dog.jpg"
img_path = "sample_dog.jpg"
if not os.path.exists(img_path):
    urllib.request.urlretrieve(img_url, img_path)

# 4. Define the preprocessing pipeline
# The original AlexNet expects 224x224 cropped images normalized with ImageNet stats
preprocess = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406], 
        std=[0.229, 0.224, 0.225]
    ),
])

# 5. Preprocess the image and prepare batch dimension
input_image = Image.open(img_path)
input_tensor = preprocess(input_image)
input_batch = input_tensor.unsqueeze(0)  # Shape becomes [1, 3, 224, 224]

# Move to GPU if available
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Running on device: {device}")
model = model.to(device)
input_batch = input_batch.to(device)

# 6. Run inference
with torch.no_grad():
    output = model(input_batch)

# Convert output scores to probabilities
probabilities = torch.nn.functional.softmax(output[0], dim=0)

# 7. Print the top 5 results
top5_prob, top5_catid = torch.topk(probabilities, 5)
print("\n--- Top 5 Predictions ---")
for i in range(top5_prob.size(0)):
    print(f"{categories[top5_catid[i]]}: {top5_prob[i].item() * 100:.2f}%")

Loading AlexNet model...
Downloading: "https://download.pytorch.org/models/alexnet-owt-7be5be79.pth" to /root/.cache/torch/hub/checkpoints/alexnet-owt-7be5be79.pth


100%|██████████| 233M/233M [00:01<00:00, 178MB/s]  


Running on device: cuda

--- Top 5 Predictions ---
Samoyed: 72.45%
wallaby: 13.94%
Pomeranian: 5.87%
Angora: 2.28%
Arctic fox: 1.25%


* **Analysis:** The model cleanly identifies the test image as a *Samoyed* with $72.45\%$ confidence. The surrounding probability distribution tails into closely matched visual features (Pomeranian, Arctic fox), indicating that the pre-trained weights and normalization matrices match standard ImageNet statistics perfectly.

### Phase 2: Inference Benchmarking

In [2]:
import time
import torch
import torchvision.models as models

# 1. Setup device (GPU is highly recommended for meaningful benchmarks)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Benchmarking on device: {device}")

# 2. Load model and set to evaluation mode
model = models.alexnet(weights=models.AlexNet_Weights.DEFAULT).to(device)
model.eval()

# 3. Create dummy input batch (e.g., batch size of 32, 3 channels, 224x224 image)
batch_size = 32
dummy_input = torch.randn(batch_size, 3, 224, 224).to(device)

# 4. Warm-up phase
# GPU initialization takes time; we discard these initial runs to get accurate data
print("Warming up the model...")
with torch.no_grad():
    for _ in range(10):
        _ = model(dummy_input)
if device.type == "cuda":
    torch.cuda.synchronize()

# 5. Actual Benchmark Loop
num_iterations = 100
total_time = 0.0

print(f"Running benchmark for {num_iterations} iterations...")
with torch.no_grad():
    for i in range(num_iterations):
        if device.type == "cuda":
            torch.cuda.synchronize()
            
        start_time = time.perf_counter()
        
        # Run inference
        _ = model(dummy_input)
        
        if device.type == "cuda":
            torch.cuda.synchronize()  # Wait for GPU operation to finish
            
        end_time = time.perf_counter()
        total_time += (end_time - start_time)

# 6. Calculate and print metrics
avg_batch_time_ms = (total_time / num_iterations) * 1000
avg_per_image_time_ms = avg_batch_time_ms / batch_size
throughput = (batch_size * num_iterations) / total_time

print("\n--- Benchmark Results ---")
print(f"Average Batch Inference Time: {avg_batch_time_ms:.2f} ms")
print(f"Average Per-Image Inference Time: {avg_per_image_time_ms:.2f} ms")
print(f"Throughput: {throughput:.2f} images/second")

Benchmarking on device: cuda
Warming up the model...
Running benchmark for 100 iterations...

--- Benchmark Results ---
Average Batch Inference Time: 8.87 ms
Average Per-Image Inference Time: 0.28 ms
Throughput: 3607.99 images/second


* **Analysis:** Processing a full batch of 32 images takes a minuscule **$8.87\text{ ms}$**, yielding an inference capability of **$3,607.99\text{ images/second}$**. The sub-millisecond per-image time ($0.28\text{ ms}$) proves that a modern runtime pipeline handles this legacy structure near instantly.

### Phase 3: Live Training Throughput

In [3]:
import time
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms
import torchvision.models as models

# 1. Setup Device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Benchmarking training on: {device}")

# 2. Data Pipeline (Upscaling to 224x224 to preserve original AlexNet math)
transform = transforms.Compose([
    transforms.Resize(224),
    transforms.ToTensor(),
    transforms.Normalize((0.4914, 0.4822, 0.4465), (0.2023, 0.1994, 0.2010)),
])

# Loads to memory instantly - bypasses Kaggle's 20GB disk bottleneck
print("Loading dataset...")
trainset = torchvision.datasets.CIFAR10(root='./data', train=True, download=True, transform=transform)
trainloader = torch.utils.data.DataLoader(trainset, batch_size=128, shuffle=True, num_workers=4, pin_memory=True)

# 3. Initialize Model (Modified final layer for 10 classes)
model = models.alexnet(weights=None) 
model.classifier[6] = nn.Linear(4096, 10)
model = model.to(device)

criterion = nn.CrossEntropyLoss()
optimizer = optim.SGD(model.parameters(), lr=0.01, momentum=0.9)

# 4. 5-Epoch Training Benchmark
num_epochs = 5
total_images = 0
start_benchmark_time = time.perf_counter()

print("\n--- Starting Benchmark Run ---")
model.train()
for epoch in range(num_epochs):
    epoch_start = time.perf_counter()
    running_loss = 0.0
    images_this_epoch = 0
    
    for inputs, labels in trainloader:
        inputs, labels = inputs.to(device), labels.to(device)
        
        optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        
        images_this_epoch += inputs.size(0)
    
    if device.type == "cuda":
        torch.cuda.synchronize()
        
    epoch_end = time.perf_counter()
    epoch_time = epoch_end - epoch_start
    epoch_throughput = images_this_epoch / epoch_time
    total_images += images_this_epoch
    
    print(f"Epoch {epoch+1}/{num_epochs} | Time: {epoch_time:.2f}s | Speed: {epoch_throughput:.2f} images/sec")

end_benchmark_time = time.perf_counter()
total_duration = end_benchmark_time - start_benchmark_time

# 5. Final Metrics Report
avg_training_throughput = total_images / total_duration
print("\n--- Final Performance Metrics ---")
print(f"Total Benchmark Time: {total_duration:.2f} seconds")
print(f"Sustained Training Throughput: {avg_training_throughput:.2f} images/second")

Benchmarking training on: cuda
Loading dataset...


100%|██████████| 170M/170M [05:13<00:00, 544kB/s] 



--- Starting Benchmark Run ---
Epoch 1/5 | Time: 51.39s | Speed: 973.01 images/sec
Epoch 2/5 | Time: 51.29s | Speed: 974.88 images/sec
Epoch 3/5 | Time: 51.08s | Speed: 978.85 images/sec
Epoch 4/5 | Time: 51.19s | Speed: 976.74 images/sec
Epoch 5/5 | Time: 51.09s | Speed: 978.69 images/sec

--- Final Performance Metrics ---
Total Benchmark Time: 256.04 seconds
Sustained Training Throughput: 976.42 images/second


* **Analysis:** Over a 5-epoch training duration, the throughput stabilizes consistently at **$976.42\text{ images/second}$**. 
* **The Training-Inference Gap:** Training takes longer than inference because a full backpropagation step requires a forward pass, a backward gradient evaluation pass (mathematically worth roughly $2\times$ the forward FLOP footprint), and a weight update optimizer step. Mathematically:
  $$\text{Training Throughput (Ideal)} \approx \frac{\text{Inference Throughput}}{3} = \frac{3607.99}{3} \approx 1202.66 \text{ images/sec}$$
* **Efficiency Factor:** The actual performance achieved ($976.42\text{ images/sec}$) reflects **$81.1\%$** of that theoretical limit. The minor loss is normal overhead: CPU data handling pipelines actively resizing images to $224 \times 224$ on-the-fly and framework spin-up taxes during the first epoch.

---

## 3. Theoretical vs. Real-World Compute Alignment

AlexNet requires approximately **$0.71 \times 10^9\text{ FLOPs}$ (Giga-FLOPs)** of floating-point computations per individual forward pass. 

Multiplying your real-world inference throughput against the workload per image reveals the actual computing performance extracted from the hardware:
$$\text{Achieved Compute} = 3607.99 \text{ images/sec} \times (0.71 \times 10^9 \text{ FLOPs}) \approx 2.56 \times 10^{12} \text{ FLOPs/sec} \quad (\mathbf{2.56\text{ TFLOPs}})$$

The hardware platform used, an **NVIDIA Tesla T4**, boasts a peak theoretical FP32 performance of **$8.1\text{ TFLOPs}$**. Dividing the metrics indicates the system ran at **$31.6\%$ absolute hardware efficiency**. 

Given that AlexNet contains severe memory-bandwidth bottlenecks—specifically its legacy fully-connected classifier layers containing over 50 million parameters that constantly choke the GPU memory bus—exceeding $30\%$ raw hardware utilization at a low batch size of 32 is highly optimal.

---

## 4. Generational Leap: 2012 vs. Today

The architectural evolution becomes glaringly clear when looking at the historical reality established in the seminal paper: 
> **Reference:** Krizhevsky, A., Sutskever, I., and Hinton, G. E. (2012). [ImageNet Classification with Deep Convolutional Neural Networks](https://papers.nips.cc/paper/4824-imagenet-classification-with-deep-convolutional-neural-networks.pdf). *NeurIPS 2012*.

### Hardware Specification Breakdown

| Metric | 2012 Original System (2x GTX 580) | Modern Evaluation System (Active: 1x Tesla T4) |
| :--- | :--- | :--- |
| **Silicon Architecture** | Fermi (40 nm) | Turing (12 nm) |
| **VRAM Footprint** | 2x 3 GB (6 GB Total) | 16 GB Single Card Used |
| **Combined FP32 Compute** | $\sim 3.16\text{ TFLOPs}$ | $\sim 8.10\text{ TFLOPs}$ (Single Active Node) |
| **Original 90-Epoch Train Time**| **5 to 6 Days** | **~23 Hours (Estimated from local run)** |

### Throughput Scaling Math
In 2012, the authors trained on 1.28 million ImageNet samples for 90 epochs over roughly 5.5 days, resulting in a training throughput of **$\sim 242.6\text{ images/second}$**.

Converting your modern local metrics back into equivalent benchmarks shows the structural advancement:
$$\text{Speedup Factor} = \frac{\text{Local Training Speed (976.42)}}{\text{Original 2012 System Speed (242.6)}} \approx \mathbf{4.02x \text{ Faster}}$$

Because the original 2012 code had to split the model across two physical PCIe cards due to strict 3GB VRAM limits (causing heavy communication bottlenecks), your single modern T4 card completely outpaces the historical baseline baseline using a single, unified GPU slot while pulling a fraction of the power footprint.

# The Deep-Dive Mechanics of AlexNet's LRN Layer

Local Response Normalization (LRN) is a non-learnable layer introduced in AlexNet (2012) to implement **lateral inhibition**—a biological phenomenon where an excited neuron suppresses the activity of its immediate neighbors to sharpen structural contrast.

---

## 1. The Mathematical Formula

The LRN layer processes data right after the activation function (like ReLU) using the following channel-wise equation:

$$b_{x,y}^i = \frac{a_{x,y}^i}{\left(k + \alpha \sum_{j=\max(0, i-n/2)}^{\min(N-1, i+n/2)} (a_{x,y}^j)^2\right)^\beta}$$

### Term-by-Term Breakdown:
* **$a_{x,y}^i$ (Input):** The original activation value at spatial coordinate $(x, y)$ inside the $i$-th channel.
* **$b_{x,y}^i$ (Output):** The normalized activation value at that same exact coordinate and channel.
* **$N$:** The total number of channels in that layer (e.g., 96 channels in Conv 1).
* **$j$ (Summation Index):** The loop counter variable that acts as a scanner, moving through adjacent channels to square and add their values together.
* **$n$ (Window Size):** The number of neighboring channels to sum over. In AlexNet, $n = 5$ (meaning it checks 2 channels below and 2 channels above target channel $i$).
* **$k$ (Bias Offset):** A baseline safety constant ($k = 2$) to prevent division-by-zero errors if all channels are inactive.
* **$\alpha$ (Scaling Factor):** A tuning hyperparameter ($\alpha = 0.0001$) that scales down the raw sum of squares.
* **$\beta$ (Power Exponent):** A compressive exponent ($\beta = 0.75$) that dampens the denominator’s growth so massive spikes don't completely wipe out the feature map.

---

## 2. The Multi-GPU Reality: Architectural Isolation

In 2012, AlexNet was trained across two **NVIDIA GTX 580 GPUs**, which were severely constrained by **only 3GB of VRAM each**. To make the network fit into memory, the architecture was split horizontally across the channel dimension:
* **GPU 1** handled channels `0` through `47`.
* **GPU 2** handled channels `48` through `95`.



### The Specialization Phenomenon:
Because these GPUs were physically isolated during convolutional layers (only communicating at layer 3 and the fully connected layers), they could not share feature maps. 
* By pure statistical randomness at weight initialization, the filters on **GPU 1** started out slightly more sensitive to high-frequency structural transitions (like sharp lines and edges).
* **GPU 2's** filters started out slightly more sensitive to continuous color variations and shading. 

Backpropagation reinforced these minor variations, turning GPU 1 into an **expert system for grayscale edges** and GPU 2 into an **expert system for color blobs**. 

Because of this physical hardware split, adjacent channels were not completely random; they naturally clustered into highly correlated visual families. LRN acted as a local regulator within each GPU, forcing these similar, co-located filters to compete and diversify rather than copy each other.

---

## 3. Granular Step-by-Step Mathematical Walkthrough

Let's compute the exact floating-point transformations of a single output pixel at coordinate $(28, 28)$ for **Target Channel 10 ($i = 10$)** using AlexNet's exact parameters ($k=2, n=5, \alpha=0.0001, \beta=0.75$).

The sliding window for $i=10$ runs from channel $j = 10 - 2 = 8$ to $j = 10 + 2 = 12$.

### Scenario A: The Unique Feature Spike (High Contrast)
Channel 10 detects a sharp structural corner, while all surrounding channels found absolutely nothing.
* **Inputs:** $a^8=0.0, \quad a^9=0.0, \quad \mathbf{a^{10}=100.0}, \quad a^{11}=0.0, \quad a^{12}=0.0$

1. **Compute the Sum of Squares:**
   $$\text{Sum} = (0.0)^2 + (0.0)^2 + (100.0)^2 + (0.0)^2 + (0.0)^2 = 10,000.0$$

2. **Compute the Denominator Base:**
   $$\text{Base} = k + (\alpha \times \text{Sum}) = 2 + (0.0001 \times 10,000.0) = 2 + 1.0 = 3.0$$

3. **Compute the Final Denominator:**
   $$\text{Denominator} = 3.0^{0.75} \approx 2.279507$$

4. **Compute Outputs for Every Channel ($b^i = a^i / \text{Denominator}$):**
   * **Target Channel 10:** $b^{10} = 100.0 / 2.279507 = \mathbf{43.869}$
   * **Neighbor Channels (8, 9, 11, 12):** $b^{\text{neigh}} = 0.0 / 2.279507 = \mathbf{0.0}$

5. **Final Output Contrast Gap:**
   $$\text{Output Gap} = b^{10} - b^{\text{neigh}} = 43.869 - 0.0 = \mathbf{+43.869}$$

---

### Scenario B: The High Uniform Block (Blinding Glare)
The target input value is identical ($100.0$), but it is buried inside a uniform, flat glare where all adjacent channels are firing at maximum capacity.
* **Inputs:** $a^8=100.0, \quad a^9=100.0, \quad \mathbf{a^{10}=100.0}, \quad a^{11}=100.0, \quad a^{12}=100.0$

1. **Compute the Sum of Squares:**
   $$\text{Sum} = (100.0)^2 + (100.0)^2 + (100.0)^2 + (100.0)^2 + (100.0)^2 = 50,000.0$$

2. **Compute the Denominator Base:**
   $$\text{Base} = k + (\alpha \times \text{Sum}) = 2 + (0.0001 \times 50,000.0) = 2 + 5.0 = 7.0$$

3. **Compute the Final Denominator:**
   $$\text{Denominator} = 7.0^{0.75} \approx 4.303517$$

4. **Compute Outputs for Every Channel ($b^i = a^i / \text{Denominator}$):**
   * **Target Channel 10:** $b^{10} = 100.0 / 4.303517 = \mathbf{23.236}$
   * **Neighbor Channels (8, 9, 11, 12):** $b^{\text{neigh}} = 100.0 / 4.303517 = \mathbf{23.236}$

5. **Final Output Contrast Gap:**
   $$\text{Output Gap} = b^{10} - b^{\text{neigh}} = 23.236 - 23.236 = \mathbf{0.0}$$

---

## 4. Comprehensive Contrast Gap Analysis



| Scenario | Input Target ($a^{10}$) | Input Neighbors ($a^{\text{neigh}}$) | Raw Input Gap | Output Target ($b^{10}$) | Output Neighbors ($b^{\text{neigh}}$) | Normalized Output Gap | Downstream Impact |
| :--- | :---: | :---: | :---: | :---: | :---: | :---: | :--- |
| **Extremely Loud Target (Spike)** | 100.0 | 0.0 | **+100.0** | 43.869 | 0.0 | **+43.869** | **Massive Landmark:** A distinct feature that stands out starkly against silence. |
| **Quiet Neighbors** | 8.0 | 0.5 | **+7.5** | 4.75 | 0.30 | **+4.45** | **Clear Local Winner:** Target preserves its relative structural lead over background noise. |
| **Low Uniform Block** | 1.0 | 1.0 | **0.0** | 0.59 | 0.59 | **0.0** | **Flat Ground:** Zero contrast. Flattened into low-level background texturing. |
| **High Uniform Block** | 100.0 | 100.0 | **0.0** | 23.236 | 23.236 | **0.0** | **Blinding Glare Deflated:** Strips out uninformative, uniform over-saturation completely. |
| **Loud Neighbors** | 8.0 | 45.0 | **-37.0** | 4.12 | 23.21 | **-19.09** | **Suppressed Signal:** Target is choked down because neighboring channels dominated the local area. |
| **Extremely Quiet Target (Black Hole)** | 0.1 | 100.0 | **-99.9** | 0.026 | 26.11 | **-26.084** | **Drowned Out:** Surrounding feature maps aggressively crush the weak target signal toward absolute zero. |

### The Core Mechanical Insight:
LRN is completely blind to the absolute magnitude of your signals; it evaluates **relative spatial-channel contrast**. 
* In the **Spike Scenario**, the feature is highly unique. LRN preserves its prominence, allowing it to pass into deeper layers with a massive relative lead ($\mathbf{+43.869}$) over the background. 
* In the **High Uniform Block**, the inputs contain zero structural change or edge data. LRN aggressively deflates the entire block down to a lower playing field, keeping the contrast gap perfectly flat at $\mathbf{0.0}$ so downstream filters ignore it.

---

## 5. Why LRN Was Permanently Abandoned

By 2015, architectures like VGG and ResNet completely eliminated LRN from deep learning pipelines due to three fatal design flaws:

1. **Logical Incoherence (Channel-Order Dependence):** LRN computes its sum across a sequential sliding window of channel indices ($i-2$ to $i+2$). This means its behavior depends entirely on the arbitrary layout order of your filters in VRAM. Shuffling the memory indices of your filters changes the mathematical outputs completely, which is structurally nonsensical since the underlying visual features haven't changed.
2. **Inability to Address Internal Covariate Shift:** LRN scales activations locally within a single forward pass of an isolated image. It does nothing to stabilize changing statistical distributions across the entire dataset as the network updates over multiple training epochs.
3. **The Efficiency of Batch Normalization (BatchNorm):** BatchNorm replaced LRN by abandoning cross-channel competition entirely. Instead of forcing unrelated filters to fight, BatchNorm normalizes **each channel independently across the mini-batch dimension**:

$$\hat{x} = \frac{x - \mu_{\mathcal{B}}}{\sqrt{\sigma_{\mathcal{B}}^2 + \epsilon}}$$

This standardizes data scaling across the whole training set, provides a lightweight regularizer, and eliminates the heavy computational overhead of calculating complex channel-wise fractional powers ($\beta=0.75$) on hardware.